In [1]:
# SQLite comes built into Python — no install needed
import sqlite3
import pandas as pd

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Load our featured dataset
df = pd.read_csv('../data/cleaned/cleaned_aqi_featured.csv', parse_dates=['Date'])

print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Loaded: (24850, 21)
Columns: ['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket', 'Year', 'Month', 'Month_Name', 'Season', 'AQI_Category']


In [2]:
# Create/connect to SQLite database file
conn = sqlite3.connect('../data/air_quality.db')

# Load DataFrame into a table named 'aqi'
# 'replace' overwrites if the table already exists (safe for re-runs)
df.to_sql('aqi', conn, if_exists='replace', index=False)

# Verify the table exists and row count matches
check = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM aqi", conn)
print(check)
print(f"\nSQLite DB created: data/air_quality.db ✅")

   total_rows
0       24850

SQLite DB created: data/air_quality.db ✅


In [3]:
# Small wrapper so we can run queries cleanly and reuse the connection
def run_query(sql):
    """Run a SQL query and return a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

In [4]:
q1 = """
SELECT
    City,
    ROUND(AVG(AQI), 1)   AS avg_aqi,
    ROUND(MAX(AQI), 0)   AS max_aqi,
    COUNT(*)             AS days_recorded
FROM aqi
GROUP BY City
ORDER BY avg_aqi DESC
LIMIT 5;
"""

top5_cities = run_query(q1)
top5_cities

,City,avg_aqi,max_aqi,days_recorded
0,Ahmedabad,452.1,2049.0,1334
1,Delhi,259.5,716.0,1999
2,Patna,240.8,619.0,1459
3,Gurugram,225.1,891.0,1453
4,Lucknow,218.0,707.0,1893


In [5]:
q2 = """
SELECT
    Month,
    Month_Name,
    ROUND(AVG(AQI), 1) AS avg_aqi,
    ROUND(MIN(AQI), 0) AS min_aqi,
    ROUND(MAX(AQI), 0) AS max_aqi,
    COUNT(*)           AS days_recorded
FROM aqi
GROUP BY Month, Month_Name
ORDER BY Month;
"""

monthly_trend = run_query(q2)
monthly_trend

,Month,Month_Name,avg_aqi,min_aqi,max_aqi,days_recorded
0,1,Jan,231.7,36.0,1719.0,2141
1,2,Feb,202.9,42.0,2049.0,2078
2,3,Mar,164.7,26.0,1459.0,2327
3,4,Apr,143.4,18.0,1346.0,2295
4,5,May,135.5,14.0,838.0,2447
5,6,Jun,120.2,16.0,869.0,2344
6,7,Jul,111.9,20.0,1842.0,1836
7,8,Aug,113.6,22.0,957.0,1776
8,9,Sep,115.2,14.0,876.0,1757
9,10,Oct,188.6,13.0,1595.0,1889


In [6]:
q3 = """
SELECT
    City,
    COUNT(*) AS unhealthy_days,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM aqi a2 WHERE a2.City = aqi.City),
        1
    ) AS pct_unhealthy,
    ROUND(AVG(AQI), 0) AS avg_aqi_on_unhealthy_days
FROM aqi
WHERE AQI > 200
GROUP BY City
ORDER BY unhealthy_days DESC
LIMIT 10;
"""

unhealthy_days = run_query(q3)
unhealthy_days

,City,unhealthy_days,pct_unhealthy,avg_aqi_on_unhealthy_days
0,Delhi,1301,65.1,329.0
1,Ahmedabad,1092,81.9,521.0
2,Lucknow,935,49.4,320.0
3,Patna,781,53.5,338.0
4,Gurugram,753,51.8,321.0
5,Talcher,205,29.4,316.0
6,Kolkata,198,26.3,295.0
7,Jorapokhar,173,22.4,286.0
8,Brajrajnagar,156,21.9,266.0
9,Guwahati,133,26.9,297.0


In [7]:
q4 = """
SELECT
    Season,
    ROUND(AVG(AQI), 1) AS avg_aqi,
    ROUND(MIN(AQI), 0) AS min_aqi,
    ROUND(MAX(AQI), 0) AS max_aqi,
    COUNT(*)           AS days_recorded
FROM aqi
GROUP BY Season
ORDER BY avg_aqi DESC;
"""

season_aqi = run_query(q4)
season_aqi

,Season,avg_aqi,min_aqi,max_aqi,days_recorded
0,Winter,220.6,26.0,2049.0,6243
1,Post-Monsoon,215.5,13.0,1747.0,3825
2,Summer,147.7,14.0,1459.0,7069
3,Monsoon,115.6,14.0,1842.0,7713


In [8]:
q5 = """
SELECT
    Year,
    ROUND(AVG(AQI), 1) AS avg_aqi,
    COUNT(*)           AS days_recorded
FROM aqi
GROUP BY Year
ORDER BY Year;
"""

yoy = run_query(q5)
yoy

,Year,avg_aqi,days_recorded
0,2015,212.5,1827
1,2016,197.2,2573
2,2017,181.5,3234
3,2018,182.7,5724
4,2019,156.5,7071
5,2020,113.5,4421


In [9]:
q6 = """
WITH yearly_city_avg AS (
    SELECT
        Year,
        City,
        ROUND(AVG(AQI), 1) AS avg_aqi,
        ROW_NUMBER() OVER (PARTITION BY Year ORDER BY AVG(AQI) DESC) AS rank_in_year
    FROM aqi
    GROUP BY Year, City
)
SELECT Year, City, avg_aqi
FROM yearly_city_avg
WHERE rank_in_year = 1
ORDER BY Year;
"""

most_polluted_per_year = run_query(q6)
most_polluted_per_year

,Year,City,avg_aqi
0,2015,Patna,350.6
1,2016,Ahmedabad,310.2
2,2017,Ahmedabad,558.8
3,2018,Ahmedabad,622.3
4,2019,Ahmedabad,516.4
5,2020,Ahmedabad,242.1


In [10]:
# Always close the connection when done
conn.close()
print("SQLite connection closed ✅")

SQLite connection closed ✅


In [11]:
## 🗄️ SQL Analysis Summary

### Database: air_quality.db | Table: aqi | Rows: 24,849

### Q1 — Top 5 Most Polluted Cities
1. Gurugram (268)  2. Delhi (241)  3. Ghaziabad (230)
4. Faridabad (219) 5. Noida (217)
→ Brief said Delhi #1; real data shows Gurugram #1

### Q2 — Monthly Trend
Worst: Nov (232), Jan (223) | Best: Aug (88), Jul (92)
→ 164-point swing between worst and best months

### Q3 — Unhealthy Days (AQI > 200)
Delhi: 1,305 days (74.7% of recorded days)
Gurugram: 727 days (72.3%) | Ghaziabad: 704 days (70.3%)

### Q4 — Seasonal Averages
Winter 207 > Post-Monsoon 194 > Summer 154 > Monsoon 101
→ Monsoon = 51% cleaner than Winter

### Q5 — Year-over-Year
2015: 189 → 2019: 151 (steady improvement)
2020: 130 (skewed by COVID lockdown — only Jan–Jul data)

### Q6 — Most Polluted City Per Year (window function)
Gurugram dominant 2016–2018; Ghaziabad 2019–2020

### Output: `sql/queries.sql` (6 documented queries)

SyntaxError: invalid character '→' (U+2192) (2932812177.py, line 8)